# Lab 3: Pruning a Neural Network Model Using TensorFlow Model Optimization Toolkit

### Objective:
The goal of this lab is to apply weight pruning on a simple dense neural network using the 
TensorFlow Model Optimization Toolkit, and evaluate its effect on model complexity, size, and accuracy.

### Pre-requisites:
- Completion of Lab 1 (Trained MNIST model)
- Familiarity with neural networks and TensorFlow
- TensorFlow Model Optimization Toolkit installed

Imp: For Prunning, tensorflow-model-optimization package is needed and this is not compatible with tensorflow new version so you need to install tf_keras and set the set TF_USE_LEGACY_KERAS=1. To do this write the below command in your terminal. <br>
`pip install tf_keras` <br>
`set TF_USE_LEGACY_KERAS=1`

In [24]:
# Step 1: Install and Import Required Libraries
!pip install tensorflow-model-optimization

In [25]:
import tensorflow as tf
import tensorflow_model_optimization as tfmot
from tensorflow import keras
import numpy as np

In [26]:
# Step 2: Load the MNIST Dataset
mnist = keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize the images
x_train = x_train / 255.0
x_test = x_test / 255.0

In [27]:
# Step 3: Define and Train a Baseline Model
def create_model():
    model = keras.Sequential([
        keras.layers.Flatten(input_shape=(28, 28)),
        keras.layers.Dense(128, activation='relu'),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(10, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

model = create_model()
model.fit(x_train, y_train, epochs=3, validation_split=0.1)

Epoch 1/3
1688/1688 [==============================] - 4s 2ms/step - loss: 0.2492 - accuracy: 0.9267 - val_loss: 0.1122 - val_accuracy: 0.9693
Epoch 2/3
1688/1688 [==============================] - 4s 2ms/step - loss: 0.1050 - accuracy: 0.9675 - val_loss: 0.0901 - val_accuracy: 0.9742
Epoch 3/3
1688/1688 [==============================] - 4s 3ms/step - loss: 0.0746 - accuracy: 0.9762 - val_loss: 0.0785 - val_accuracy: 0.9788


In [28]:
# Step 4: Apply Weight Pruning
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.5,
        begin_step=0,
        end_step=np.ceil(x_train.shape[0] / 128).astype(np.int32) * 3
    )
}

pruned_model = tfmot.sparsity.keras.prune_low_magnitude(model, **pruning_params)

pruned_model.compile(optimizer='adam',
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

# Add pruning callbacks
callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

# Fine-tune the pruned model
pruned_model.fit(x_train, y_train,
                 batch_size=128,
                 epochs=3,
                 validation_split=0.1,
                 callbacks=callbacks)

Epoch 1/3
422/422 [==============================] - 3s 4ms/step - loss: 0.0416 - accuracy: 0.9870 - val_loss: 0.0792 - val_accuracy: 0.9792
Epoch 2/3
422/422 [==============================] - 2s 5ms/step - loss: 0.0296 - accuracy: 0.9914 - val_loss: 0.0711 - val_accuracy: 0.9803
Epoch 3/3
422/422 [==============================] - 2s 4ms/step - loss: 0.0225 - accuracy: 0.9942 - val_loss: 0.0735 - val_accuracy: 0.9788


In [29]:
# Step 5: Strip Pruning Wrappers and Save the Model
model_for_export = tfmot.sparsity.keras.strip_pruning(pruned_model)
model_for_export.save("mnist_pruned_model.keras")

In [30]:
# Convert the pruned model to TFLite
model = keras.models.load_model("mnist_pruned_model.keras")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
pruned_tflite = converter.convert()
with open("mnist_pruned_model.tflite", "wb") as f:
    f.write(pruned_tflite)

print("Pruned model saved as mnist_pruned_model.tflite")

INFO:tensorflow:Assets written to: C:\Users\bhawa\AppData\Local\Temp\tmph11fsl2c\assets


INFO:tensorflow:Assets written to: C:\Users\bhawa\AppData\Local\Temp\tmph11fsl2c\assets


Pruned model saved as mnist_pruned_model.tflite


In [31]:
# Use a small test set for evaluation
x_test_sample = x_test[:100]
y_test_sample = y_test[:100]

In [32]:
# Evaluate the Pruned Model
interpreter = tf.lite.Interpreter(model_path="mnist_pruned_model.tflite")
interpreter.allocate_tensors()

input_index = interpreter.get_input_details()[0]['index']
output_index = interpreter.get_output_details()[0]['index']

correct = 0
for i in range(len(x_test_sample)):
    input_data = np.expand_dims(x_test_sample[i], axis=0).astype(np.float32)
    interpreter.set_tensor(input_index, input_data)
    interpreter.invoke()
    output = interpreter.get_tensor(output_index)
    pred = np.argmax(output)
    if pred == y_test_sample[i]:
        correct += 1

accuracy = correct / len(x_test_sample)
print(f"Pruned Model Accuracy on 100 samples: {accuracy * 100:.2f}%")

Pruned Model Accuracy on 100 samples: 100.00%


C:\Users\bhawa\AppData\Roaming\Python\Python311\site-packages\tensorflow\lite\python\interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
